## Cell 1 — Install + imports
`awesome_cossim_topn` (the v0.* API you were using) is **deprecated and slated for removal** — see the ["Migrating to v1"](https://pypi.org/project/sparse-dot-topn/) section of the package's own docs. We switch to the current `sp_matmul_topn` API, which also uses a max-heap internally (`O(top_n)` instead of `O(n_columns)` for the top-n step) and is reported up to 6x faster in the package's own benchmarks — so this is a free win, not just a compatibility fix.

In [ ]:
!pip install anyascii "sparse-dot-topn>=1.0" -q

import os, gc, re, time
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anyascii
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

print("Libraries successfully imported!")

## Cell 2 — Paths (train AND test)
Same auto-detection as before, but now resolving **both** train and test paths — Cell 6 needs `train_ground_truth.tsv` to validate a threshold before we trust it on the real test set.

In [ ]:
def find_data_dirs():
    roots = ["/kaggle/input", "/kaggle/input/amazon-ml-challenge-2026", "student_resource/dataset", "."]
    t_dir, te_dir = None, None
    for r in roots:
        if not os.path.exists(r):
            continue
        for dp, _, files in os.walk(r):
            fset = set(files)
            if "test_source1.tsv" in fset and "test_source2.tsv" in fset:
                te_dir = dp
            if "train_source1.tsv" in fset and "train_ground_truth.tsv" in fset:
                t_dir = dp
    return t_dir, te_dir

TRAIN_DIR, TEST_DIR = find_data_dirs()
OUTPUT_DIR = "/kaggle/working/output"
PARTIAL_DIR = "/kaggle/working/partial_test"     # per-country checkpoints for the test run
VAL_PARTIAL_DIR = "/kaggle/working/partial_val"  # per-country checkpoints for validation
for d in (OUTPUT_DIR, PARTIAL_DIR, VAL_PARTIAL_DIR):
    os.makedirs(d, exist_ok=True)

print(f"TRAIN_DIR : {TRAIN_DIR}")
print(f"TEST_DIR  : {TEST_DIR}")

test_paths = {
    "s1": os.path.join(TEST_DIR, "test_source1.tsv"),
    "s2": os.path.join(TEST_DIR, "test_source2.tsv"),
    "s3": os.path.join(TEST_DIR, "test_source3.tsv"),
}
train_paths = {
    "s1": os.path.join(TRAIN_DIR, "train_source1.tsv"),
    "s2": os.path.join(TRAIN_DIR, "train_source2.tsv"),
    "s3": os.path.join(TRAIN_DIR, "train_source3.tsv"),
    "gt": os.path.join(TRAIN_DIR, "train_ground_truth.tsv"),
}
for label, paths in [("test", test_paths), ("train", train_paths)]:
    for k, p in paths.items():
        assert os.path.exists(p), f"Missing required {label} file: {p}"
        print(f"  [OK] {label}.{k}: {os.path.getsize(p)/(1024*1024):.1f} MB -> {p}")

## Cell 3 — Text normalization
Unchanged from your version (still `anyascii`, prioritizing blocking speed as you intended). One note added below on `uroman` as an optional later A/B test, not a default.

In [ ]:
LEGAL_RE = re.compile(
    r'\b(pvt|private|ltd|limited|inc|incorporated|corp|corporation|llc|llp|co|company|sarl|sas|sa|sci|eurl|snc)\b',
    re.IGNORECASE
)
PUNCT_RE = re.compile(r'[^a-z0-9\s]')
SPACE_RE = re.compile(r'\s+')

def normalize_text_list(texts: list) -> list:
    cleaned = []
    for raw in texts:
        if not raw or pd.isna(raw):
            cleaned.append("")
            continue
        t = anyascii.anyascii(str(raw)).lower()
        t = LEGAL_RE.sub(" ", t)
        t = PUNCT_RE.sub(" ", t)
        t = SPACE_RE.sub(" ", t).strip()
        cleaned.append(t)
    return cleaned

def build_composite_text(df: pd.DataFrame) -> list:
    names = df["business_name"].fillna("").astype(str).tolist()
    addrs = df["business_address"].fillna("").astype(str).tolist()
    combined = [f"{n} {a}".strip() for n, a in zip(names, addrs)]
    return normalize_text_list(combined)

# NOTE: if the recall ceiling in Cell 7 looks short, the highest-leverage swap is `uroman`
# for the non-ASCII share of rows (India especially) -- it's real transliteration, not just
# character stripping, and it only costs anything on rows that actually need it. Worth an
# A/B test later, just not a default here since you switched away from LaBSE for speed.

## Cell 4 — Resumable, per-country blocking
Two changes from your version:
1. **Random sample for the TF-IDF fit**, not the first N rows — avoids bias if the source file has any natural ordering (by city, by segment, etc.).
2. **Writes each country straight to its own file** and skips instantly if that file already exists. A crash on India no longer costs you the US partition you already finished.

In [ ]:
EMPTY_CAND_DF = pd.DataFrame({"entity_id": [], "business_name": [], "business_address": [], "country": []})

def block_one_country(
    country: str,
    sub_s1: pd.DataFrame,
    sub_cand: pd.DataFrame,
    out_path: str,
    k: int = 25,
    threshold: float = 0.10,
    chunk_size: int = 50_000,
    sample_fit_size: int = 250_000,
    n_threads: int = 4,
    seed: int = 42,
):
    # Blocks one country partition and writes source1_entity_id / candidate_entity_ids /
    # scores straight to `out_path`. Skips entirely if out_path already exists and is non-empty.
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"  [{country}] {out_path} already exists -- skipping (delete it to force a rerun)")
        return

    n_queries, n_cands = len(sub_s1), len(sub_cand)
    print(f"  [{country}] Queries: {n_queries:,} | Candidate Pool: {n_cands:,}")

    if n_queries == 0:
        open(out_path, "w").close()
        return

    if n_cands == 0:
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("source1_entity_id\tcandidate_entity_ids\tscores\n")
            for sid in sub_s1["entity_id"]:
                f.write(f"{sid}\t\t\n")
        print(f"  [{country}] no candidates in this country -- wrote all-singleton file")
        return

    t0 = time.time()
    s1_texts = build_composite_text(sub_s1)
    cand_texts = build_composite_text(sub_cand)
    print(f"  [{country}] normalization: {time.time()-t0:.1f}s")

    t_fit = time.time()
    rng = np.random.default_rng(seed)
    if len(cand_texts) > sample_fit_size:
        fit_idx = rng.choice(len(cand_texts), size=sample_fit_size, replace=False)
        fit_sample = [cand_texts[i] for i in fit_idx]
    else:
        fit_sample = cand_texts
    vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 4), max_features=120_000,
        sublinear_tf=True, norm="l2", dtype=np.float32,
    )
    vec.fit(fit_sample)
    print(f"  [{country}] TF-IDF fit: {time.time()-t_fit:.1f}s (vocab={len(vec.vocabulary_):,})")
    del fit_sample

    t_tr = time.time()
    mat_cand = vec.transform(cand_texts)
    cand_T = mat_cand.T.tocsr()
    del cand_texts, mat_cand
    gc.collect()
    print(f"  [{country}] candidate matrix: {time.time()-t_tr:.1f}s | nnz={cand_T.nnz:,}")

    s1_ids = sub_s1["entity_id"].to_numpy()
    cand_ids = sub_cand["entity_id"].to_numpy()

    tmp_path = out_path + ".partial"
    t_srch = time.time()
    with open(tmp_path, "w", encoding="utf-8") as f:
        f.write("source1_entity_id\tcandidate_entity_ids\tscores\n")
        for start in range(0, n_queries, chunk_size):
            end = min(start + chunk_size, n_queries)
            mat_q = vec.transform(s1_texts[start:end])
            # sp_matmul_topn is the current (non-deprecated) API:
            #   ntop -> top_n, lower_bound -> threshold, use_threads/n_jobs -> n_threads
            sim = sp_matmul_topn(mat_q, cand_T, top_n=k, threshold=threshold, n_threads=n_threads, sort=True)
            counts = np.diff(sim.indptr)
            offset = 0
            for i, cnt in enumerate(counts):
                sid = s1_ids[start + i]
                if cnt == 0:
                    f.write(f"{sid}\t\t\n")
                else:
                    row_cands = cand_ids[sim.indices[offset:offset+cnt]]
                    row_scores = sim.data[offset:offset+cnt]
                    f.write(f"{sid}\t{','.join(row_cands.tolist())}\t{','.join(f'{s:.4f}' for s in row_scores)}\n")
                offset += cnt
            del mat_q, sim
    os.rename(tmp_path, out_path)
    print(f"  [{country}] search: {time.time()-t_srch:.1f}s | total: {time.time()-t0:.1f}s")
    del s1_texts, cand_T, vec
    gc.collect()

## Cell 5 — Run blocking on the FULL test set
One change worth calling out: the candidate pool is grouped by country **once**, then the big unfiltered ~10M-row frame is deleted immediately. Your version kept it alive for the whole country loop (re-filtering it each time via `cand_df[cand_df["country"]==c]`), which is fine for compute but means the full pool sits in RAM the entire time — on top of everything else, that's real headroom given Kaggle's ~30GB ceiling.

In [ ]:
print("Loading test_source1...")
test_s1 = pd.read_csv(test_paths["s1"], sep="\t", dtype=str)
test_s1["country"] = test_s1["country"].str.strip()
print(f"  {len(test_s1):,} rows")
print(test_s1["country"].value_counts())

print("\nLoading test candidate pool (S2+S3)...")
t0 = time.time()
test_s2 = pd.read_csv(test_paths["s2"], sep="\t", dtype=str)
test_s3 = pd.read_csv(test_paths["s3"], sep="\t", dtype=str)
test_cand_pool = pd.concat([test_s2, test_s3], ignore_index=True)
test_cand_pool["country"] = test_cand_pool["country"].str.strip()
del test_s2, test_s3
gc.collect()
print(f"  {len(test_cand_pool):,} candidates loaded in {time.time()-t0:.1f}s")

cand_by_country = {c: g.reset_index(drop=True) for c, g in test_cand_pool.groupby("country")}
del test_cand_pool
gc.collect()
s1_by_country = {c: g.reset_index(drop=True) for c, g in test_s1.groupby("country")}

for country in sorted(test_s1["country"].dropna().unique()):
    print(f"\n=== TEST -- {country} ===")
    block_one_country(
        country=country,
        sub_s1=s1_by_country[country],
        sub_cand=cand_by_country.get(country, EMPTY_CAND_DF),
        out_path=os.path.join(PARTIAL_DIR, f"test_{country}.tsv"),
    )

del cand_by_country, s1_by_country
gc.collect()
print("\nAll test country partitions blocked -- see partial_test/ for per-country checkpoints.")

## Cell 6 — Validate against TRAIN ground truth
This is the piece that was missing before: `PRECISION_THRESHOLD = 0.45` in your version was asserted, not measured. This runs the same blocking function on a **sample** of train S1 queries (index-build cost per country is set by the candidate pool size either way, so sampling only shrinks the cheap query-side cost) and checks it against `train_ground_truth.tsv`.

In [ ]:
VALIDATION_SAMPLE_PER_COUNTRY = 15_000
VAL_SEED = 7

print("Loading train_source1 + ground truth...")
train_s1_full = pd.read_csv(train_paths["s1"], sep="\t", dtype=str)
train_s1_full["country"] = train_s1_full["country"].str.strip()
gt = pd.read_csv(train_paths["gt"], sep="\t", dtype=str, keep_default_na=False)
gt_map = {
    r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x)
    for r in gt.itertuples(index=False)
}

rng = np.random.default_rng(VAL_SEED)
sampled_ids = []
for country, grp in train_s1_full.groupby("country"):
    n = min(VALIDATION_SAMPLE_PER_COUNTRY, len(grp))
    sampled_ids.extend(rng.choice(grp["entity_id"].to_numpy(), size=n, replace=False))
val_s1 = train_s1_full[train_s1_full["entity_id"].isin(sampled_ids)].reset_index(drop=True)
print(f"  Validation sample: {len(val_s1):,} S1 entities across {val_s1['country'].nunique()} countries")
del train_s1_full
gc.collect()

print("\nLoading + grouping train candidate pool (same size class as test)...")
train_s2 = pd.read_csv(train_paths["s2"], sep="\t", dtype=str)
train_s3 = pd.read_csv(train_paths["s3"], sep="\t", dtype=str)
train_cand_pool = pd.concat([train_s2, train_s3], ignore_index=True)
train_cand_pool["country"] = train_cand_pool["country"].str.strip()
del train_s2, train_s3
gc.collect()
train_cand_by_country = {c: g.reset_index(drop=True) for c, g in train_cand_pool.groupby("country")}
del train_cand_pool
gc.collect()

for country in sorted(val_s1["country"].dropna().unique()):
    print(f"\n=== VALIDATION -- {country} ===")
    block_one_country(
        country=country,
        sub_s1=val_s1[val_s1["country"] == country].reset_index(drop=True),
        sub_cand=train_cand_by_country.get(country, EMPTY_CAND_DF),
        out_path=os.path.join(VAL_PARTIAL_DIR, f"val_{country}.tsv"),
    )
del train_cand_by_country
gc.collect()

## Cell 7 — Score it: recall ceiling + threshold sweep
Two numbers you didn't have before:
- **Recall ceiling** — the fraction of true matches that even *appear* in the raw candidate list, before any threshold. This is the hard cap on your final score; the problem statement calls this out explicitly ("blocking determines the upper bound of your recall").
- **Macro F0.5 per threshold** — an actual, data-driven choice for the cutoff instead of an asserted 0.45.

In [ ]:
def load_all_partials(directory, prefix):
    rows = {}
    for fn in sorted(os.listdir(directory)):
        if not (fn.startswith(prefix) and fn.endswith(".tsv")):
            continue
        df = pd.read_csv(os.path.join(directory, fn), sep="\t", dtype=str, keep_default_na=False)
        for r in df.itertuples(index=False):
            cids = [x for x in r.candidate_entity_ids.split(",") if x]
            scores = [float(x) for x in r.scores.split(",") if x]
            rows[r.source1_entity_id] = list(zip(cids, scores))
    return rows

def f_beta(true_ids, pred_ids, beta=0.5):
    true_ids, pred_ids = set(true_ids), set(pred_ids)
    if not true_ids:
        return 1.0 if not pred_ids else 0.0
    if not pred_ids:
        return 0.0
    tp = len(true_ids & pred_ids)
    p = tp / len(pred_ids)
    r = tp / len(true_ids)
    if p == 0 and r == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * p * r / (b2 * p + r)

val_results = load_all_partials(VAL_PARTIAL_DIR, "val_")
val_ids = list(val_results.keys())

recall_hits, recall_total = 0, 0
for sid in val_ids:
    true = gt_map.get(sid, set())
    if not true:
        continue
    all_cands = {cid for cid, _ in val_results[sid]}
    recall_hits += len(true & all_cands)
    recall_total += len(true)
print(f"Blocking recall ceiling (k=25, before any threshold): "
      f"{recall_hits}/{recall_total} = {recall_hits/max(recall_total,1):.3f}")
print("(a match the blocker never retrieved can't be recovered by any threshold")
print(" or later matching model -- this is your real ceiling)")

print(f"\n{'threshold':>10} | {'macro F0.5':>10}")
best_tau, best_f = None, -1.0
for tau in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    scores = [f_beta(gt_map.get(sid, set()), [cid for cid, sc in val_results[sid] if sc >= tau])
              for sid in val_ids]
    macro_f = float(np.mean(scores))
    print(f"{tau:>10.2f} | {macro_f:>10.4f}")
    if macro_f > best_f:
        best_f, best_tau = macro_f, tau

print(f"\nBest threshold on validation sample: tau={best_tau} (macro F0.5={best_f:.4f})")

## Cell 8 — Assemble the final submission files
Uses `best_tau` from Cell 7 if it ran, otherwise falls back to 0.45 so this cell still works standalone.

In [ ]:
FINAL_THRESHOLD = best_tau if "best_tau" in dir() else 0.45
print(f"Using precision threshold tau = {FINAL_THRESHOLD}")

test_results = load_all_partials(PARTIAL_DIR, "test_")
all_test_ids = test_s1["entity_id"].tolist()

cand_file = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
match_file = os.path.join(OUTPUT_DIR, "matching_results.tsv")

with open(cand_file, "w", encoding="utf-8") as fc, open(match_file, "w", encoding="utf-8") as fm:
    fc.write("source1_entity_id\tcandidate_entity_ids\n")
    fm.write("source1_entity_id\tmatched_entity_ids\n")
    n_singleton = 0
    for sid in all_test_ids:
        pairs = test_results.get(sid, [])
        fc.write(f"{sid}\t{','.join(cid for cid, _ in pairs)}\n")
        matched = [cid for cid, sc in pairs if sc >= FINAL_THRESHOLD]
        if not matched:
            n_singleton += 1
        fm.write(f"{sid}\t{','.join(matched)}\n")

print(f"Wrote {cand_file} and {match_file}")
print(f"  {n_singleton:,}/{len(all_test_ids):,} predicted as singletons "
      f"({n_singleton/len(all_test_ids):.1%})")

## Cell 9 — Local validation suite
Same checks as your version. One change: it rebuilds the candidate-ID set by reading just the `entity_id` column from disk rather than keeping the full ~10M-row candidate DataFrame alive from Cell 5 just for this check.

In [ ]:
def run_local_validation(matching_path, candidate_path, test_s1_path, test_cand_ids):
    errors = []
    print("\nRunning comprehensive validation checks...")

    s1_ids = pd.read_csv(test_s1_path, sep="\t", dtype=str)["entity_id"].tolist()
    expected_rows = len(s1_ids)

    df_m = pd.read_csv(matching_path, sep="\t", dtype=str).fillna("")
    df_c = pd.read_csv(candidate_path, sep="\t", dtype=str).fillna("")

    if len(df_m) != expected_rows:
        errors.append(f"matching_results.tsv row count mismatch: {len(df_m)} vs {expected_rows}")
    if len(df_c) != expected_rows:
        errors.append(f"candidate_pairs.tsv row count mismatch: {len(df_c)} vs {expected_rows}")

    if list(df_m.columns) != ["source1_entity_id", "matched_entity_ids"]:
        errors.append(f"Invalid columns in matching_results: {list(df_m.columns)}")
    if list(df_c.columns) != ["source1_entity_id", "candidate_entity_ids"]:
        errors.append(f"Invalid columns in candidate_pairs: {list(df_c.columns)}")

    m_dict = dict(zip(df_m["source1_entity_id"], df_m["matched_entity_ids"]))
    c_dict = dict(zip(df_c["source1_entity_id"], df_c["candidate_entity_ids"]))

    for s1 in s1_ids:
        m_list = [x.strip() for x in m_dict.get(s1, "").split(",") if x.strip()]
        c_list = [x.strip() for x in c_dict.get(s1, "").split(",") if x.strip()]

        if len(m_list) != len(set(m_list)):
            errors.append(f"Duplicate IDs in match list for {s1}"); break
        if len(c_list) != len(set(c_list)):
            errors.append(f"Duplicate IDs in candidate list for {s1}"); break
        if not set(m_list).issubset(set(c_list)):
            errors.append(f"Match ID for {s1} is not present in its candidate list!"); break
        for cid in c_list:
            if not (cid.startswith("S2-") or cid.startswith("S3-")):
                errors.append(f"Candidate {cid} does not have S2- or S3- prefix"); break
            if cid not in test_cand_ids:
                errors.append(f"Entity {cid} does not exist in candidate pool!"); break
        if errors:
            break

    if not errors:
        print("=" * 60)
        print("STATUS: PASS (EXIT 0) -- READY FOR PORTAL SUBMISSION!")
        print("=" * 60)
    else:
        print("STATUS: FAILED")
        for err in errors:
            print(f"  [ERROR] {err}")

_test_cand_ids = set()
for p in (test_paths["s2"], test_paths["s3"]):
    _test_cand_ids.update(pd.read_csv(p, sep="\t", dtype=str, usecols=["entity_id"])["entity_id"])

run_local_validation(
    matching_path=match_file,
    candidate_path=cand_file,
    test_s1_path=test_paths["s1"],
    test_cand_ids=_test_cand_ids,
)

official_script = "student_resource/utils/validate_submission.py"
if os.path.exists(official_script):
    get_ipython().system('python3 {official_script} --matching {match_file} --candidate {cand_file} --test-dir {TEST_DIR}')